# 확률과 확률분포

경우의 수를 나누는 방식은 기본 결과들이 같은 확률일 때 사용합니다. 확률변수의 값 종류(이산/연속)와 확률 배분 규칙(균등/이항/정규/포아송)은 서로 다른 구분입니다.

**학습 목표:** 표본공간·사건·조건부확률, 이산균등분포와 기대값, 이항분포, 포아송분포, 연속분포와 구간 확률

공식 → 직접 구현 또는 계산 → 라이브러리 대조 → 해석 순서로 학습합니다.

## 실행 준비

Python 3.11 이상, NumPy·pandas·SciPy·Matplotlib이 필요합니다. 없는 경우 별도 셀에서 `%pip install numpy pandas scipy matplotlib`을 한 번 실행하세요. 새 커널에서 위에서 아래로 실행합니다. 외부 데이터 다운로드는 없습니다. 그래프는 폰트 호환을 위해 영문 제목을 쓰고 해석은 한국어로 제공합니다.

**표기:** 기본 기술 분산은 ddof=0, 모집단 분산 추정·평균 추론은 ddof=1을 명시합니다. 난수는 seed=42(문제에 지정 시 해당 seed)입니다.

In [1]:
import math
from collections import Counter
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
# 외부 파일 없이 재현 가능한 학습용 가상 자료
x = np.array([12, 15, 14, 10, 18, 20, 22, 25, 30, 28], dtype=float)
y = np.array([2, 4, 6, 8, 10, 12, 14, 40], dtype=float)
plt.rcParams['axes.unicode_minus'] = False


## 1. 표본공간·사건·조건부확률

두 공정한 독립 동전의 표본공간은 HH,HT,TH,TT입니다. 앞뒤가 같을 확률은 2/4입니다. 조건부확률 P(A|B)=P(A∩B)/P(B)에서 P(B)>0이어야 합니다. 독립이면 P(A∩B)=P(A)P(B)입니다.

In [2]:
space=[a+b for a in 'HT' for b in 'HT']
print(space,sum(s[0]==s[1] for s in space)/len(space))

['HH', 'HT', 'TH', 'TT'] 0.5


**해석 포인트:** 각각 1/2,1/2,1/3,2/3입니다. 1/3≠1/4이므로 독립이 아닙니다.

**직접 확인:** 대응 실습의 문제 1에서 다른 입력으로 적용해 보세요.

## 2. 이산균등분포와 기대값

공정한 주사위는 1~6이 각각 1/6입니다. 기대값은 ΣxP(x), 확률의 합은 1입니다. 기대값이 실제 가능한 값일 필요는 없습니다.

In [3]:
v=np.arange(1,7);p=np.ones(6)/6
print(np.sum(v*p),stats.randint.mean(1,7))
assert np.isclose(p.sum(),1)

3.5 3.5


**해석 포인트:** 기대값 1.1, 분산 .49입니다. 여기서 확률 가중 분산은 표본분산이 아니므로 n−1로 나누지 않습니다.

**직접 확인:** 대응 실습의 문제 2에서 다른 입력으로 적용해 보세요.

## 3. 이항분포

횟수 n이 고정되고 성공확률 p가 일정한 독립 시행의 성공 횟수입니다. P(X=k)=조합(n,k)p^k(1−p)^(n−k). 독립성이나 동일 확률 가정이 깨지면 이항모형이 부적절할 수 있습니다.

In [4]:
def binom_manual(n,k,p):
    return math.comb(n,k)*p**k*(1-p)**(n-k)
print(binom_manual(10,3,.2),stats.binom.pmf(3,10,.2))
assert np.isclose(binom_manual(10,3,.2),stats.binom.pmf(3,10,.2))

0.2013265920000001 0.201326592


**해석 포인트:** 정확히 2명 약 .2852, 1명 이상 약 .8784, 기대 인원 2명입니다. sf(0)는 P(X>0)로 누적확률의 보완입니다.

**직접 확인:** 대응 실습의 문제 3에서 다른 입력으로 적용해 보세요.

## 4. 포아송분포

일정 시간의 사건 수에 사용합니다. 독립적 발생과 일정 발생률을 가정합니다. 평균 λ이면 분산도 λ이고, 시간 단위를 바꾸면 λ도 바뀝니다.

In [5]:
def poisson_manual(k,lam): return math.exp(-lam)*lam**k/math.factorial(k)
print(poisson_manual(2,3),stats.poisson.pmf(2,3))
assert np.isclose(poisson_manual(2,3),stats.poisson.pmf(2,3))

0.22404180765538775 0.22404180765538775


**해석 포인트:** 30분의 λ는 2입니다. 확률은 약 .1353과 .3233입니다. 업무 시간별 발생률이 크게 다르면 일정률 가정을 다시 확인합니다.

**직접 확인:** 대응 실습의 문제 4에서 다른 입력으로 적용해 보세요.

## 5. 연속분포와 구간 확률

연속변수는 점 하나의 확률이 0이며 밀도 높이와 확률은 다릅니다. 구간 확률은 누적분포함수 CDF의 차 또는 밀도곡선 아래 면적입니다.

In [6]:
print(stats.uniform.cdf(6,loc=0,scale=10)-stats.uniform.cdf(2,loc=0,scale=10))
print(stats.norm.cdf(40,35,5)-stats.norm.cdf(30,35,5))

0.39999999999999997
0.6826894921370859


**해석 포인트:** 각각 약 .1587입니다. 평균이 같아도 분산과 늦은 배달 꼬리 확률이 다르면 마감시간을 지킬 가능성도 달라집니다. 정규모형은 음수 시간도 허용하므로 실제 데이터 적합성을 확인해야 합니다.

**직접 확인:** 대응 실습의 문제 5에서 다른 입력으로 적용해 보세요.

## 복습 순서

1. 관측 대상과 통계량의 의미를 말로 설명합니다.
2. 분모·단위·독립성·분포 가정을 확인합니다.
3. 직접 계산과 라이브러리 결과를 비교합니다.
4. 코드 결과와 실제로 주장할 수 있는 범위를 구분합니다.

**학습 이동:** `Statistics_3_probability.ipynb` → `Statistics_3_probability_Practice.ipynb` → `Statistics_3_probability_Answer.ipynb`.